Sound features extraction

In [ ]:
import os

from os import listdir
from os.path import isfile, join

import pandas as pd

from glob import glob

import librosa
import numpy as np

import matplotlib.pyplot as plt

from scipy.signal import hilbert, chirp

import scipy

import math

In [ ]:
#Data loading from drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Sound Data and vibration damage assessment data loading
data_dir_ini = '/content/drive/MyDrive/Article sound/audio/4_norm/'
audio_files_ini = glob(data_dir_ini+'/*.wav')

df = pd.read_table("/content/drive/MyDrive/Article sound/data/raw_data_FT_Vibration_damage_assessment.csv", sep=',' , encoding='utf-8-sig')


In [ ]:
df

,sound_sample_name,sample,form,geol,hit_number,sample_name,sample_number,sound_sample_name_simple,sound_number,f1_ini,a1_ini,f1_0,a1_0,f1_n,a1_n
0,L_pl_a_1,L,pl,a,1,S01,1,S01_1,1,2748.0,0.37,2590.0,0.73,2499.76,1.02
1,L_pl_a_2,L,pl,a,2,S01,1,S01_2,2,2748.0,0.37,2590.0,0.73,2499.76,1.02
2,L_pl_a_3,L,pl,a,3,S01,1,S01_3,3,2748.0,0.37,2590.0,0.73,2499.76,1.02
3,L_pl_a_4,L,pl,a,4,S01,1,S01_4,4,2748.0,0.37,2590.0,0.73,2499.76,1.02
4,L_pl_a_5,L,pl,a,5,S01,1,S01_5,5,2748.0,0.37,2590.0,0.73,2499.76,1.02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,P_pl_a_5,P,pl,a,5,S31,31,S31_5,5,3020.0,0.43,2943.0,0.39,2854.71,0.37
161,P_pl_a_6,P,pl,a,6,S31,31,S31_6,6,3020.0,0.43,2943.0,0.39,2854.71,0.37
162,P_pl_a_7,P,pl,a,7,S31,31,S31_7,7,3020.0,0.43,2943.0,0.39,2854.71,0.37
163,P_pl_a_8,P,pl,a,8,S31,31,S31_8,8,3020.0,0.43,2943.0,0.39,2854.71,0.37


Features assessment

In [ ]:
#fpeak assessment
def compute_fpeak_audio(y, sr, n_fft, sound_name):
    # Compute F1 as the frequency with maximum intensity
    spectrum = abs(np.fft.fft(y))[0: int(len(y) / 2)]
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    f1 = freqs[np.argmax(spectrum)]

    # Plots
    plt.plot(freqs[:-1], spectrum)
    plt.title(sound_name)
    plt.legend(["fpeak: " + '%.f' % f1])
    plt.show()
    print(sound_name, '\t', '%.f' % f1)


    return(f1)

In [ ]:
# Sound decay rate assessment

# Function to determine the signal envelope by finding local peaks
def find_peaks_window(x, y, window):
    peak_x = []
    peak_vals = []
    n = len(y)
    for i in range(window, n - window):
        if (y[i] > np.max(y[i-window:i])) and (y[i] > np.max(y[i+1:i+1+window])):
            peak_x.append(x[i])
            peak_vals.append(y[i])
    return np.array(peak_x), np.array(peak_vals)

# Mono-exponential decay function for the envelope
def monoExp(x, m, c, b):
    return m * np.exp(-c * x) + b

# Fit the envelope and compute the damping coefficient
def compute_damping(y, sr, sound_name):
    time = np.arange(len(y)) / sr

    peak_x, peak_vals = find_peaks_window(time, y, window=20)

    m0 = 1
    b0 = 0
    c0 = 1.0 / (peak_x[1] - peak_x[0])

    # Perform the curve fitting
    p0 = (m0, c0, b0)
    params, cv = scipy.optimize.curve_fit(monoExp, peak_x, peak_vals, p0, maxfev=2000)
    m, c, b = params

    # Determine the quality of the fit (Coefficient of Determination - R²)
    squaredDiffs = np.square(peak_vals - monoExp(peak_x, m, c, b))
    squaredDiffsFromMean = np.square(peak_vals - np.mean(peak_vals))
    rSquared = 1 - np.sum(squaredDiffs) / np.sum(squaredDiffsFromMean)
    print(f"R² = {rSquared}")

    # Print the fitted equation parameters
    print(f"Y = {m} * e^(-{c} * x) + {b}")

    plt.title(sound_name)

    plt.plot(time, y, zorder=1)

    plt.scatter(peak_x, peak_vals, color='green', label="Pics détectés")
    plt.plot(peak_x, monoExp(peak_x, m, c, b), '--', label="fitted", linewidth=2, zorder=10)

    plt.legend(f"R² = {rSquared}")

    plt.show()

    return c

In [ ]:
# audio features extracted from librosa

audio_files_ini.sort()

for file in range(0, len(audio_files_ini),1):

    path = audio_files_ini[file]

    name = os.path.basename(path)
    name_simple = name.replace("_norm.wav", "")

    y, sr = librosa.load(path, sr=None)
    frame_lenght = len(y)
    row_index = df.loc[(df == name_simple).any(axis=1)].index[0]

    print('------------------------')
    print('\n', name_simple)

    plt.plot(np.arange(len(y)) / sr, y)
    plt.title(name_simple)
    plt.show()


    df.loc[row_index, "sound_path"] = path

    rms = librosa.feature.rms(y=y, S=None, frame_length=frame_lenght, center=False)[0][0]
    df.loc[row_index, "rms"] = rms

    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=frame_lenght, center=False, window='boxcar')[0][0]
    df.loc[row_index, "centroid"] = centroid

    flatness = librosa.feature.spectral_flatness(y=y, S=None, n_fft=frame_lenght, center=False, window='boxcar')[0][0]

    df.loc[row_index, "flatness"] = flatness

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, n_fft=frame_lenght, center=False, window='boxcar')[0][0]

    df.loc[row_index, "rolloff"] = rolloff

    contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_fft=frame_lenght, center=False, n_bands=1, window='boxcar')[0][0]

    df.loc[row_index, "contrast"] = contrast

    f1 = compute_fpeak_audio(y, sr, frame_lenght, name_simple)

    df.loc[row_index, "fpeak"] = f1

    c = compute_damping(y, sr, name_simple)
    df.loc[row_index, "c"] = c

    print("RMS: ", rms)
    print("Centroid: ", centroid)
    print("Flatness: ", flatness)
    print("Rolloff: ", rolloff)
    print("Contrast: ", contrast)
    print("fpeak: ", f1)
    print("c: ", c)






Output hidden; open in https://colab.research.google.com to view.

In [ ]:
df

,sound_sample_name,sample,form,geol,hit_number,sample_name,sample_number,sound_sample_name_simple,sound_number,f1_ini,...,f1_n,a1_n,sound_path,rms,centroid,flatness,rolloff,contrast,fpeak,c
0,L_pl_a_1,L,pl,a,1,S01,1,S01_1,1,2748.0,...,2499.76,1.02,/content/drive/MyDrive/Article sound/audio/4_n...,0.122204,5841.668799,0.008963,12397.290432,14.673802,3223.793395,55.140871
1,L_pl_a_2,L,pl,a,2,S01,1,S01_2,2,2748.0,...,2499.76,1.02,/content/drive/MyDrive/Article sound/audio/4_n...,0.129421,6028.623430,0.007643,14052.751905,10.566528,3227.942422,59.485896
2,L_pl_a_3,L,pl,a,3,S01,1,S01_3,3,2748.0,...,2499.76,1.02,/content/drive/MyDrive/Article sound/audio/4_n...,0.111234,6446.748089,0.012436,16177.053345,12.863068,3223.793395,57.890860
3,L_pl_a_4,L,pl,a,4,S01,1,S01_4,4,2748.0,...,2499.76,1.02,/content/drive/MyDrive/Article sound/audio/4_n...,0.128124,6010.692139,0.008403,13538.272650,14.208599,3227.942422,63.182687
4,L_pl_a_5,L,pl,a,5,S01,1,S01_5,5,2748.0,...,2499.76,1.02,/content/drive/MyDrive/Article sound/audio/4_n...,0.130879,6893.026751,0.013390,16525.571550,19.863773,3227.942422,60.316329
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,P_pl_a_5,P,pl,a,5,S31,31,S31_5,5,3020.0,...,2854.71,0.37,/content/drive/MyDrive/Article sound/audio/4_n...,0.186941,6135.595823,0.003788,13310.076207,16.549231,3456.138865,26.039233
161,P_pl_a_6,P,pl,a,6,S31,31,S31_6,6,3020.0,...,2854.71,0.37,/content/drive/MyDrive/Article sound/audio/4_n...,0.174533,5758.063079,0.003231,9463.928874,16.699080,3456.138865,26.775503
162,P_pl_a_7,P,pl,a,7,S31,31,S31_7,7,3020.0,...,2854.71,0.37,/content/drive/MyDrive/Article sound/audio/4_n...,0.129668,4726.509981,0.000583,6634.292972,20.380640,3451.989839,26.909441
163,P_pl_a_8,P,pl,a,8,S31,31,S31_8,8,3020.0,...,2854.71,0.37,/content/drive/MyDrive/Article sound/audio/4_n...,0.201567,5464.058606,0.002583,8414.225233,13.928079,3456.138865,24.830045


Damage evaluation and label creation

In [ ]:
# 1. Final damage calculation: D = 1 - (f1_n² / f1_0²)
df["D"] = 1 - (df["f1_n"] ** 2) / (df["f1_0"] ** 2)

# 2. Quality factor calculation (Initial vibration quality factor)
df["Q_vibration_initial"] = (1 / (2 * df["a1_ini"])) * 100

# Frost resistance classification for threshold D < 0.1
df["frost_resistant_01"] = (df["D"] < 0.1).astype(int)


In [ ]:
df

,sound_sample_name,sample,form,geol,hit_number,sample_name,sample_number,sound_sample_name_simple,sound_number,f1_ini,...,rms,centroid,flatness,rolloff,contrast,fpeak,c,D,Q_vibration_initial,frost_resistant_01
0,L_pl_a_1,L,pl,a,1,S01,1,S01_1,1,2748.0,...,0.122204,5841.668799,0.008963,12397.290432,14.673802,3223.793395,55.140871,0.068469,135.135135,1
1,L_pl_a_2,L,pl,a,2,S01,1,S01_2,2,2748.0,...,0.129421,6028.623430,0.007643,14052.751905,10.566528,3227.942422,59.485896,0.068469,135.135135,1
2,L_pl_a_3,L,pl,a,3,S01,1,S01_3,3,2748.0,...,0.111234,6446.748089,0.012436,16177.053345,12.863068,3223.793395,57.890860,0.068469,135.135135,1
3,L_pl_a_4,L,pl,a,4,S01,1,S01_4,4,2748.0,...,0.128124,6010.692139,0.008403,13538.272650,14.208599,3227.942422,63.182687,0.068469,135.135135,1
4,L_pl_a_5,L,pl,a,5,S01,1,S01_5,5,2748.0,...,0.130879,6893.026751,0.013390,16525.571550,19.863773,3227.942422,60.316329,0.068469,135.135135,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,P_pl_a_5,P,pl,a,5,S31,31,S31_5,5,3020.0,...,0.186941,6135.595823,0.003788,13310.076207,16.549231,3456.138865,26.039233,0.059100,116.279070,1
161,P_pl_a_6,P,pl,a,6,S31,31,S31_6,6,3020.0,...,0.174533,5758.063079,0.003231,9463.928874,16.699080,3456.138865,26.775503,0.059100,116.279070,1
162,P_pl_a_7,P,pl,a,7,S31,31,S31_7,7,3020.0,...,0.129668,4726.509981,0.000583,6634.292972,20.380640,3451.989839,26.909441,0.059100,116.279070,1
163,P_pl_a_8,P,pl,a,8,S31,31,S31_8,8,3020.0,...,0.201567,5464.058606,0.002583,8414.225233,13.928079,3456.138865,24.830045,0.059100,116.279070,1


Exportation des données par son en CSV

In [ ]:
# Exporter en CSV sans l’index
df.to_csv('/content/drive/MyDrive/Article sound/data/sound_features.csv', index=False, encoding='utf-8')

Exportation des données par échantillon

In [ ]:
frost_resistant_01 = (df.groupby("sample_name")["frost_resistant_01"]
          .apply(list)
          .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
          .add_prefix("frost_resitant_0"))

Q_vibration_initial = (df.groupby("sample_name")["Q_vibration_initial"]
            .apply(list)
            .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
            .add_prefix("Q_vibration_initial"))

form = (df.groupby("sample_name")["form"]
            .apply(list)
            .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
            .add_prefix("form_"))

centroid = (df.groupby("sample_name")["centroid"]
          .apply(list)
          .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
          .add_prefix("centroid_"))

rms = (df.groupby("sample_name")["rms"]
          .apply(list)
          .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
          .add_prefix("rms_"))

flatness = (df.groupby("sample_name")["flatness"]
            .apply(list)
            .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
            .add_prefix("flatness_"))

rolloff = (df.groupby("sample_name")["rolloff"]
          .apply(list)
          .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
          .add_prefix("rolloff_"))

contrast = (df.groupby("sample_name")["contrast"]
            .apply(list)
            .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
            .add_prefix("contrast_"))

fpeak = (df.groupby("sample_name")["fpeak"]
            .apply(list)
            .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
            .add_prefix("fpeak_"))

c = (df.groupby("sample_name")["c"]
          .apply(list)
          .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
          .add_prefix("c_"))

out = pd.concat([frost_resistant_01["frost_resitant_01"], Q_vibration_initial["Q_vibration_initial1"], form["form_1"], rms, flatness, rolloff, contrast, fpeak, centroid, c], axis=1).reset_index()

out = out.rename(columns={
    "frost_resitant_01": "frost_resistance",
    "Q_vibration_initial1": "Q_vibration_initial",
    "form_1": "form",
})
out = out.fillna("")

print(out)

   sample_name  frost_resistance  Q_vibration_initial form     rms_1  \
0          S01               1.0           135.135135   pl  0.122204   
1          S02               1.0           119.047619   pl  0.118736   
2          S03               1.0            81.967213   pl  0.105056   
3          S04               1.0           125.000000   pt  0.103743   
4          S05               1.0           131.578947   pt  0.106428   
5          S06               1.0           116.279070   pt  0.102041   
6          S07               1.0           111.111111   pt  0.117247   
7          S08               1.0            79.365079   pt  0.150835   
8          S09               1.0           131.578947   pt   0.14204   
9          S10               1.0            60.240964   pt  0.139613   
10         S11               0.0            96.153846   pl   0.10425   
11         S12               0.0            40.983607   pl  0.100455   
12         S13               1.0           147.058824   pl  0.10

In [ ]:
out

,sample_name,frost_resistance,Q_vibration_initial,form,rms_1,rms_2,rms_3,rms_4,rms_5,rms_6,...,centroid_9,c_1,c_2,c_3,c_4,c_5,c_6,c_7,c_8,c_9
0,S01,1.0,135.135135,pl,0.122204,0.129421,0.111234,0.128124,0.130879,0.114371,...,4881.061628,55.140871,59.485896,57.89086,63.182687,60.316329,66.255863,59.274832,64.133336,58.61262
1,S02,1.0,119.047619,pl,0.118736,0.145612,0.109484,0.137154,0.138425,0.13242,...,5688.567321,53.590213,55.768092,52.344184,52.534777,56.854063,54.397365,52.550665,48.540497,56.479053
2,S03,1.0,81.967213,pl,0.105056,0.118863,0.093317,0.106551,0.100342,0.09387,...,4705.888714,56.806662,44.97485,59.088341,51.434448,58.481212,62.897749,44.583643,54.349873,55.834852
3,S04,1.0,125.000000,pt,0.103743,0.107076,0.106194,,,,...,,104.057411,103.448224,96.625126,,,,,,
4,S05,1.0,131.578947,pt,0.106428,0.103779,0.109662,,,,...,,95.872463,95.976631,86.70923,,,,,,
5,S06,1.0,116.279070,pt,0.102041,0.102716,0.102248,,,,...,,110.161899,104.201227,103.432419,,,,,,
6,S07,1.0,111.111111,pt,0.117247,0.121782,,,,,...,,68.654684,67.570562,,,,,,,
7,S08,1.0,79.365079,pt,0.150835,0.161233,0.144541,,,,...,,43.730744,42.526667,43.138527,,,,,,
8,S09,1.0,131.578947,pt,0.14204,0.136088,0.133269,,,,...,,55.755492,55.517256,53.960332,,,,,,
9,S10,1.0,60.240964,pt,0.139613,0.143177,0.131833,,,,...,,51.011316,50.166816,55.983247,,,,,,


In [ ]:
stats = out.copy()

# Pop column in string format
sample_name = stats.pop('sample_name')
form = stats.pop('form')

# Convert other columns in float
stats = stats.apply(pd.to_numeric)

# Compute stats (mean, std, min, max) for each features

rms_columns = ['rms_1', 'rms_2', 'rms_3', 'rms_4', 'rms_5', 'rms_6', 'rms_7', 'rms_8', 'rms_9']
stats['rms_mean'] = stats[rms_columns].mean(axis=1)
stats['rms_std'] = stats[rms_columns].std(axis=1)
stats['rms_min'] = stats[rms_columns].min(axis=1)
stats['rms_max'] = stats[rms_columns].max(axis=1)
stats['rms_range'] = stats['rms_max'] - stats['rms_min']

flatness_columns = ['flatness_1', 'flatness_2', 'flatness_3', 'flatness_4', 'flatness_5', 'flatness_6', 'flatness_7', 'flatness_8', 'flatness_9']
stats['flatness_mean'] = stats[flatness_columns].mean(axis=1)
stats['flatness_std'] = stats[flatness_columns].std(axis=1)
stats['flatness_min'] = stats[flatness_columns].min(axis=1)
stats['flatness_max'] = stats[flatness_columns].max(axis=1)
stats['flatness_range'] = stats['flatness_max'] - stats['flatness_min']


rolloff_columns = ['rolloff_1', 'rolloff_2', 'rolloff_3', 'rolloff_4', 'rolloff_5', 'rolloff_6', 'rolloff_7', 'rolloff_8', 'rolloff_9']
stats['rolloff_mean'] = stats[rolloff_columns].mean(axis=1)
stats['rolloff_std'] = stats[rolloff_columns].std(axis=1)
stats['rolloff_min'] = stats[rolloff_columns].min(axis=1)
stats['rolloff_max'] = stats[rolloff_columns].max(axis=1)
stats['rolloff_range'] = stats['rolloff_max'] - stats['rolloff_min']

contrast_columns = ['contrast_1', 'contrast_2', 'contrast_3', 'contrast_4', 'contrast_5', 'contrast_6', 'contrast_7', 'contrast_8', 'contrast_9']
stats['contrast_mean'] = stats[contrast_columns].mean(axis=1)
stats['contrast_std'] = stats[contrast_columns].std(axis=1)
stats['contrast_min'] = stats[contrast_columns].min(axis=1)
stats['contrast_max'] = stats[contrast_columns].max(axis=1)
stats['contrast_range'] = stats['contrast_max'] - stats['contrast_min']

fpeak_columns = ['fpeak_1', 'fpeak_2', 'fpeak_3', 'fpeak_4', 'fpeak_5', 'fpeak_6', 'fpeak_7', 'fpeak_8', 'fpeak_9']
stats['fpeak_mean'] = stats[fpeak_columns].mean(axis=1)
stats['fpeak_std'] = stats[fpeak_columns].std(axis=1)
stats['fpeak_min'] = stats[fpeak_columns].min(axis=1)
stats['fpeak_max'] = stats[fpeak_columns].max(axis=1)
stats['fpeak_range'] = stats['fpeak_max'] - stats['fpeak_min']

centroid_columns = ['centroid_1', 'centroid_2', 'centroid_3', 'centroid_4', 'centroid_5', 'centroid_6', 'centroid_7', 'centroid_8', 'centroid_9']
stats['centroid_mean'] = stats[centroid_columns].mean(axis=1)
stats['centroid_std'] = stats[centroid_columns].std(axis=1)
stats['centroid_min'] = stats[centroid_columns].min(axis=1)
stats['centroid_max'] = stats[centroid_columns].max(axis=1)
stats['centroid_range'] = stats['centroid_max'] - stats['centroid_min']

c_columns = ['c_1', 'c_2', 'c_3', 'c_4', 'c_5', 'c_6', 'c_7', 'c_8', 'c_9']
stats['c_mean'] = stats[c_columns].mean(axis=1)
stats['c_std'] = stats[c_columns].std(axis=1)
stats['c_min'] = stats[c_columns].min(axis=1)
stats['c_max'] = stats[c_columns].max(axis=1)
stats['c_range'] = stats['c_max'] - stats['c_min']

stats.insert(0, 'form', form)
stats.insert(0, 'sample_name', sample_name)

stats

,sample_name,form,frost_resistance,Q_vibration_initial,rms_1,rms_2,rms_3,rms_4,rms_5,rms_6,...,centroid_mean,centroid_std,centroid_min,centroid_max,centroid_range,c_mean,c_std,c_min,c_max,c_range
0,S01,pl,1.0,135.135135,0.122204,0.129421,0.111234,0.128124,0.130879,0.114371,...,6012.621609,589.024741,4881.061628,6893.026751,2011.965123,60.477033,3.449210,55.140871,66.255863,11.114992
1,S02,pl,1.0,119.047619,0.118736,0.145612,0.109484,0.137154,0.138425,0.132420,...,6911.786747,1151.963200,5358.211364,8567.668763,3209.457399,53.673212,2.587895,48.540497,56.854063,8.313566
2,S03,pl,1.0,81.967213,0.105056,0.118863,0.093317,0.106551,0.100342,0.093870,...,5459.906511,652.005642,4558.928434,6197.090933,1638.162499,54.272403,6.250442,44.583643,62.897749,18.314106
3,S04,pt,1.0,125.000000,0.103743,0.107076,0.106194,NaN,NaN,NaN,...,8189.893378,1005.291861,7279.460418,9268.764887,1989.304469,101.376921,4.126432,96.625126,104.057411,7.432285
4,S05,pt,1.0,131.578947,0.106428,0.103779,0.109662,NaN,NaN,NaN,...,7017.449919,698.488585,6278.363827,7666.635800,1388.271973,92.852775,5.320721,86.709230,95.976631,9.267401
5,S06,pt,1.0,116.279070,0.102041,0.102716,0.102248,NaN,NaN,NaN,...,7115.259415,217.196343,6869.146299,7280.095180,410.948881,105.931849,3.683444,103.432419,110.161899,6.729480
6,S07,pt,1.0,111.111111,0.117247,0.121782,NaN,NaN,NaN,NaN,...,7694.798151,634.360661,7246.237426,8143.358876,897.121450,68.112623,0.766591,67.570562,68.654684,1.084123
7,S08,pt,1.0,79.365079,0.150835,0.161233,0.144541,NaN,NaN,NaN,...,5347.441113,388.359444,5078.519540,5792.681481,714.161941,43.131979,0.602065,42.526667,43.730744,1.204076
8,S09,pt,1.0,131.578947,0.142040,0.136088,0.133269,NaN,NaN,NaN,...,5691.594200,120.390900,5596.319438,5826.901464,230.582027,55.077694,0.974967,53.960332,55.755492,1.795160
9,S10,pt,1.0,60.240964,0.139613,0.143177,0.131833,NaN,NaN,NaN,...,5955.162487,122.788945,5823.560949,6066.656816,243.095867,52.387126,3.142826,50.166816,55.983247,5.816431


In [ ]:
print(list(stats.columns.values))

['sample_name', 'form', 'frost_resistance', 'Q_vibration_initial', 'rms_1', 'rms_2', 'rms_3', 'rms_4', 'rms_5', 'rms_6', 'rms_7', 'rms_8', 'rms_9', 'flatness_1', 'flatness_2', 'flatness_3', 'flatness_4', 'flatness_5', 'flatness_6', 'flatness_7', 'flatness_8', 'flatness_9', 'rolloff_1', 'rolloff_2', 'rolloff_3', 'rolloff_4', 'rolloff_5', 'rolloff_6', 'rolloff_7', 'rolloff_8', 'rolloff_9', 'contrast_1', 'contrast_2', 'contrast_3', 'contrast_4', 'contrast_5', 'contrast_6', 'contrast_7', 'contrast_8', 'contrast_9', 'fpeak_1', 'fpeak_2', 'fpeak_3', 'fpeak_4', 'fpeak_5', 'fpeak_6', 'fpeak_7', 'fpeak_8', 'fpeak_9', 'centroid_1', 'centroid_2', 'centroid_3', 'centroid_4', 'centroid_5', 'centroid_6', 'centroid_7', 'centroid_8', 'centroid_9', 'c_1', 'c_2', 'c_3', 'c_4', 'c_5', 'c_6', 'c_7', 'c_8', 'c_9', 'rms_mean', 'rms_std', 'rms_min', 'rms_max', 'rms_range', 'flatness_mean', 'flatness_std', 'flatness_min', 'flatness_max', 'flatness_range', 'rolloff_mean', 'rolloff_std', 'rolloff_min', 'rolloff

In [ ]:
# Export to CSV without the index
stats.to_csv('/content/drive/MyDrive/Article sound/stats.csv', index=False, encoding='utf-8')